# Tier 4 — Fuzzy Matching with Medha

Medha's waterfall has five tiers. The first four (L1 cache, template, exact vector,
semantic) rely on embedding similarity. **Tier 4 — Fuzzy Matching** catches the
cases they miss:

| Input problem | Example | Why semantic fails | Why fuzzy wins |
|---|---|---|---|
| Typos | `employes` | embedding space is not typo-aware | Levenshtein edit distance is small |
| Abbreviations | `avg sal eng` | too short/ambiguous | character overlap is high |
| OCR errors | `Sh0w emp1oyees` | digit/letter confusion | ratio still ~85%+ |
| Minor rewording | `List staff in Eng.` | may fall below sem. threshold | ratio ~90%+ |

**How it works internally:**
1. A vector **pre-filter** retrieves the top-K most similar candidates from Qdrant
   (fast, ~O(log n))
2. Levenshtein `fuzz.ratio()` is applied only to those K candidates (O(K))
3. The best ratio above `score_threshold_fuzzy` (0–100) is returned

**Requirements:**
```bash
pip install "medha-archai[fastembed]" rapidfuzz
```

## Cell 1: Setup

In [ ]:
import time

try:
    import rapidfuzz  # noqa: F401
    print("rapidfuzz is available — Tier 4 fuzzy matching is enabled")
except ImportError:
    print("rapidfuzz not found. Install with: pip install rapidfuzz")
    print("Fuzzy matching cells will return NO_MATCH without it.")

from medha import Medha, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

## Cell 2: Populate the Cache

We store a set of SQL queries that will serve as the ground truth for fuzzy matching.

In [ ]:
pairs = [
    ("How many employees are there?",          "SELECT COUNT(*) FROM employees"),
    ("List all employees in Engineering",       "SELECT * FROM employees WHERE department='Engineering'"),
    ("What is the average salary in Engineering?", "SELECT AVG(salary) FROM employees WHERE department='Engineering'"),
    ("Show top 10 products by price",           "SELECT * FROM products ORDER BY price DESC LIMIT 10"),
    ("Count orders placed today",               "SELECT COUNT(*) FROM orders WHERE DATE(created_at)=CURDATE()"),
    ("List customers from New York",            "SELECT * FROM customers WHERE city='New York'"),
    ("Total revenue this quarter",              "SELECT SUM(amount) FROM orders WHERE QUARTER(created_at)=QUARTER(NOW())"),
]

# Use a low fuzzy threshold to demonstrate hits on moderately corrupted input
settings = Settings(
    backend_type="qdrant",
    qdrant_mode="memory",
    score_threshold_fuzzy=75.0,          # Default is 85; lowered for demo
    score_threshold_fuzzy_prefilter=0.4, # Wider pre-filter net
    fuzzy_prefilter_top_k=30,
)

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")
medha = Medha("fuzzy_demo", embedder=embedder, settings=settings)
await medha.start()

for question, sql in pairs:
    await medha.store(question, sql)

print(f"{len(pairs)} entries stored. Fuzzy threshold: {settings.score_threshold_fuzzy}")

## Cell 3: Typos

Users often mistype questions. Semantic similarity degrades sharply for typos because
the embedding model tokenizes at the subword level — a misspelled word often maps to
an out-of-vocabulary token with a different vector.

Fuzzy matching compares the **normalized character sequence** directly, so minor
edit distances still score above the threshold.

In [ ]:
# Clear L1 so every search goes through the full waterfall
await medha.clear_caches()

typo_queries = [
    ("How many employes are there?",          "employes → employees"),
    ("List all emplyees in Enginnering",      "emplyees + Enginnering"),
    ("Waht is the avrage salary in Engineering?", "Waht + avrage"),
    ("Count ordres placed today",             "ordres → orders"),
    ("List custmers from New York",           "custmers → customers"),
]

print(f"{'Typo query':<50s}  {'Strategy':<20s}  {'Conf':>6s}")
print("-" * 82)

for query, note in typo_queries:
    hit = await medha.search(query)
    conf = f"{hit.confidence:.3f}" if hit.confidence else "  n/a"
    print(f"{query:<50s}  {hit.strategy.value:<20s}  {conf}")
    if hit.generated_query and hit.strategy == SearchStrategy.FUZZY_MATCH:
        print(f"  → {hit.generated_query}")

## Cell 4: Abbreviations & Shorthand

Data analysts often type abbreviated questions in dashboards and BI tools.
These are too short for reliable semantic embedding but have high character overlap
with the stored questions.

In [ ]:
await medha.clear_caches()

abbrev_queries = [
    "# employees",
    "employees Engineering",
    "avg salary Engineering",
    "top 10 products price",
    "orders today count",
]

print(f"{'Abbreviated query':<40s}  {'Strategy':<20s}  {'Conf':>6s}")
print("-" * 72)

for q in abbrev_queries:
    hit = await medha.search(q)
    conf = f"{hit.confidence:.3f}" if hit.confidence else "  n/a"
    print(f"{q:<40s}  {hit.strategy.value:<20s}  {conf}")
    if hit.generated_query:
        print(f"  → {hit.generated_query[:70]}")

## Cell 5: OCR / Copy-Paste Errors

When questions arrive via OCR (e.g., from a scanned document or screenshot) or
copy-paste from a PDF, common substitutions occur: `0/O`, `1/l/I`, `rn/m`, etc.

In [ ]:
await medha.clear_caches()

ocr_queries = [
    ("H0w many emp1oyees are there?",     "0→O, 1→l"),
    ("L1st all employees in Engineering", "1→i"),
    ("Sh0w top 10 products by prlce",     "0→o, l→i in 'price'"),
    ("C0unt orders placed t0day",         "0→o twice"),
]

print(f"{'OCR-corrupted query':<52s}  {'Strategy':<20s}  {'Conf':>6s}  Note")
print("-" * 100)

for query, note in ocr_queries:
    hit = await medha.search(query)
    conf = f"{hit.confidence:.3f}" if hit.confidence else "  n/a"
    print(f"{query:<52s}  {hit.strategy.value:<20s}  {conf}  {note}")

## Cell 6: Threshold Tuning

The fuzzy threshold controls the sensitivity vs. precision trade-off:

| `score_threshold_fuzzy` | Behavior |
|---|---|
| 95+ | Only near-identical strings match (minimal false positives) |
| 85 | Default — good balance for 1–3 character typos |
| 75 | Catches more typos but risks false positives on short questions |
| <70 | Too permissive — unrelated short questions may match |

This cell sweeps the threshold to show how it affects hit rate on a controlled workload.

In [ ]:
test_cases = [
    # (corrupted_question, expected_sql, label)
    ("How many employes are there?",       "SELECT COUNT(*) FROM employees",              "1 typo"),
    ("List all emplyees in Engineering",   "SELECT * FROM employees WHERE department='Engineering'", "2 typos"),
    ("Count ordres placed today",          "SELECT COUNT(*) FROM orders WHERE DATE(created_at)=CURDATE()", "1 typo"),
    ("H0w many emp1oyees are there?",      "SELECT COUNT(*) FROM employees",              "OCR 2 subs"),
    ("Something completely unrelated xyz", None,                                           "should miss"),
]

thresholds = [95.0, 85.0, 75.0, 65.0]

print(f"  {'Threshold':>10s}  {'Hits':>5s}  {'False+':>7s}  {'Correct':>8s}")
print("  " + "-" * 40)

for threshold in thresholds:
    cfg = Settings(
        backend_type="qdrant",
        qdrant_mode="memory",
        score_threshold_fuzzy=threshold,
        score_threshold_fuzzy_prefilter=0.4,
        fuzzy_prefilter_top_k=30,
    )
    m = Medha(f"thresh_{threshold}", embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=cfg)
    await m.start()
    for q, sql in pairs:
        await m.store(q, sql)
    await m.clear_caches()

    hits = correct = false_pos = 0
    for corrupted, expected_sql, _ in test_cases:
        h = await m.search(corrupted)
        is_hit = h.strategy not in (SearchStrategy.NO_MATCH, SearchStrategy.ERROR)
        if is_hit:
            hits += 1
            if expected_sql is None:
                false_pos += 1
            elif h.generated_query == expected_sql:
                correct += 1

    await m.close()
    print(f"  {threshold:>10.1f}  {hits:>5d}  {false_pos:>7d}  {correct:>8d}")

## Cell 7: Pre-filter Tuning

The vector pre-filter (`score_threshold_fuzzy_prefilter`, `fuzzy_prefilter_top_k`)
determines how many candidates are passed to Levenshtein scoring:

- **Larger `top_k`** → higher recall (fewer misses), higher latency
- **Lower `score_threshold_fuzzy_prefilter`** → wider net, more candidates

Without the pre-filter, Medha would scroll the entire collection — O(n) instead of O(K).

In [ ]:
await medha.clear_caches()

corrupted = "How many employes are there?"

prefilter_configs = [
    (0.8, 10,  "tight prefilter"),
    (0.6, 20,  "moderate prefilter"),
    (0.4, 50,  "wide prefilter (default)"),
    (0.2, 100, "very wide prefilter"),
]

print(f"Query: {corrupted!r}\n")
print(f"  {'Config':<28s}  {'Strategy':<20s}  {'Conf':>6s}  {'Time':>8s}")
print("  " + "-" * 70)

for pf_thresh, top_k, label in prefilter_configs:
    cfg = Settings(
        backend_type="qdrant",
        qdrant_mode="memory",
        score_threshold_fuzzy=75.0,
        score_threshold_fuzzy_prefilter=pf_thresh,
        fuzzy_prefilter_top_k=top_k,
    )
    m = Medha(f"pf_{top_k}", embedder=FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache"), settings=cfg)
    await m.start()
    for q, sql in pairs:
        await m.store(q, sql)
    await m.clear_caches()

    t0 = time.perf_counter()
    h = await m.search(corrupted)
    elapsed = (time.perf_counter() - t0) * 1000

    conf = f"{h.confidence:.3f}" if h.confidence else "  n/a"
    print(f"  {label:<28s}  {h.strategy.value:<20s}  {conf}  {elapsed:6.1f}ms")
    await m.close()

## Cell 8: Cleanup

In [ ]:
await medha.close()
print("Fuzzy matching demo complete.")
print()
print("Key takeaways:")
print("  • Tier 4 fuzzy matching catches typos, abbreviations, and OCR errors")
print("  • It activates only when Tiers 0-3 all miss — zero overhead on clean input")
print("  • score_threshold_fuzzy=85 is a safe default; lower for tolerant matching")
print("  • The vector pre-filter keeps fuzzy search fast even on large collections")
print("  • Install rapidfuzz to enable this tier: pip install rapidfuzz")